In [ ]:
print("hello")

In [ ]:
!pip install -q nnsight transformers "torch>=2.4" accelerate huggingface_hub scikit-learn python-dotenv numpy tqdm matplotlib

In [ ]:
import os
import re
import torch
import gc
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt

from tqdm.auto import tqdm
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from sklearn.linear_model import LogisticRegression
from dotenv import load_dotenv
from nnsight import VisionLanguageModel

import warnings
warnings.filterwarnings("ignore")

load_dotenv()
hf_token = os.getenv('HF_TOKEN')

In [ ]:
import nnsight, transformers
print(f"torch version: {torch.__version__}")
print(f"transformers version: {transformers.__version__}")
print(f"nnsight version: {nnsight.__version__}")

In [ ]:
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
DEVICE

In [ ]:
def clear_memory():
    global model, tokenizer
    try:
        del model
        del tokenizer
        print("Model and tokenizer deleted")
    except:
        print("Model/tokenizer not found")  
    gc.collect()
    torch.cuda.empty_cache()
    free, total = torch.cuda.mem_get_info()
    print(f"GPU Memory | free: {free/1e9:.1f} GB / total: {total/1e9:.1f} GB (used: {(total-free)/1e9:.1f} GB)")

clear_memory()

In [ ]:
MODEL_ID = "google/gemma-4-E2B-it"
model = VisionLanguageModel(MODEL_ID, device_map=DEVICE, dtype=torch.float16, dispatch=True)
tokenizer = model.tokenizer

cfg = getattr(model.config, "text_config", model.config)
N_LAYERS = cfg.num_hidden_layers
D = cfg.hidden_size
print(f"layers={N_LAYERS}, hidden={D}")

In [ ]:
with model.trace("Hello world! This is a test."):
    h = model.model.language_model.layers[0].input.save()
print("residual vector shape,", h.shape) 

In [ ]:
from pathlib import Path
from google.colab import drive
drive.mount('/content/drive')

try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    DATA_ROOT = Path('/content/drive/My Drive')
    print("running on colab, drive mounted")
else:
    DATA_ROOT = Path.home() / 'Desktop' / 'told-vs-inferred'
    print("running locally")

In [ ]:
data_dirs = [
    DATA_ROOT / 'data' / 'llama_gender_1',
    DATA_ROOT / 'data' / 'llama_gender_2'
]

def parse_conversation(text):
    turns = []
    pattern = r'(HUMAN|ASSISTANT):\s*(.*?)(?=(?:HUMAN|ASSISTANT):|$)'
    matches = re.findall(pattern, text, re.DOTALL)
    for speaker, content in matches:
        turns.append({'speaker': speaker, 'text': content.strip()})
    return turns

def get_full_conversation_text(turns):
    return '\n'.join([f"{t['speaker']}: {t['text']}" for t in turns])

conversations = {}
gender_labels = {}

for data_dir in data_dirs:
    if not data_dir.exists():
        print(f"skipping {data_dir} (not found)")
        continue
    
    for file in sorted(data_dir.glob("*.txt")):
        parts = file.stem.split("_gender_")
        conv_id = int(parts[0].replace("conversation_", ""))
        gender = parts[1]
        
        with open(file, 'r') as f:
            raw_text = f.read()
        
        turns = parse_conversation(raw_text)
        full_text = get_full_conversation_text(turns)
        
        conversations[conv_id] = {
            'turns': turns,
            'full_text': full_text,
            'raw': raw_text,
            'file': file.name
        }
        gender_labels[conv_id] = gender

by_gender = {'male': {}, 'female': {}}
for conv_id, gender in gender_labels.items():
    by_gender[gender][conv_id] = conversations[conv_id]

print(f"loaded {len(conversations)} conversations")
print(f"male: {len(by_gender['male'])}, female: {len(by_gender['female'])}")

In [ ]:
# extract activations from all conversations for all layers
print("extracting activations from all conversations...\n")

X_by_layer = {i: [] for i in range(N_LAYERS)}
y_labels = []
conv_metadata = []

for conv_id in sorted(conversations.keys()):
    gender = gender_labels[conv_id]
    conv_text = conversations[conv_id]['full_text']
    
    try:
        with model.trace(conv_text):
            for layer_idx in range(N_LAYERS):
                h = model.model.language_model.layers[layer_idx].output[0].save()
                last_token = h[0, -1, :].cpu().detach().numpy()
                X_by_layer[layer_idx].append(last_token)
        
        y_labels.append(1 if gender == 'male' else 0)
        conv_metadata.append({'conv_id': conv_id, 'gender': gender})
        
    except Exception as e:
        print(f"error conv {conv_id}: {e}")

for layer_idx in range(N_LAYERS):
    X_by_layer[layer_idx] = np.array(X_by_layer[layer_idx])

y_labels = np.array(y_labels)

print(f"extracted {len(y_labels)} conversations across {N_LAYERS} layers")
print(f"layer 0 shape: {X_by_layer[0].shape}")
print(f"labels: {np.sum(y_labels):.0f} male, {len(y_labels) - np.sum(y_labels):.0f} female")

In [ ]:
X_train_by_layer = {}
X_test_by_layer = {}
y_train = None
y_test = None

for layer_idx in range(N_LAYERS):
    X_tr, X_te, y_tr, y_te = train_test_split(
        X_by_layer[layer_idx], y_labels, test_size=0.2, 
        random_state=42, stratify=y_labels
    )
    X_train_by_layer[layer_idx] = X_tr
    X_test_by_layer[layer_idx] = X_te
    if y_train is None:
        y_train = y_tr
        y_test = y_te

print(f"train: {y_train.shape[0]} ({np.sum(y_train):.0f}M, {len(y_train)-np.sum(y_train):.0f}F)")
print(f"test:  {y_test.shape[0]} ({np.sum(y_test):.0f}M, {len(y_test)-np.sum(y_test):.0f}F)")

In [ ]:
class DifferenceInMeansProbe:
    def __init__(self):
        self.direction = None
        self.threshold = None
    
    def fit(self, X, y):
        male_mask = y == 1
        female_mask = y == 0
        mean_male = X[male_mask].mean(axis=0)
        mean_female = X[female_mask].mean(axis=0)
        self.direction = mean_male - mean_female
        self.direction = self.direction / np.linalg.norm(self.direction)
        self.threshold = 0.0
    
    def predict(self, X):
        scores = X @ self.direction
        return (scores > self.threshold).astype(int)

dim_results = {'layers': [], 'train_acc': [], 'test_acc': []}

print("training difference in means probes...\n")
for layer_idx in tqdm(range(N_LAYERS)):
    probe = DifferenceInMeansProbe()
    probe.fit(X_train_by_layer[layer_idx], y_train)
    
    train_acc = accuracy_score(y_train, probe.predict(X_train_by_layer[layer_idx]))
    test_acc = accuracy_score(y_test, probe.predict(X_test_by_layer[layer_idx]))
    
    dim_results['layers'].append(layer_idx)
    dim_results['train_acc'].append(train_acc)
    dim_results['test_acc'].append(test_acc)

print(f"done | avg test acc: {np.mean(dim_results['test_acc']):.3f}")

In [ ]:
lr_results = {'layers': [], 'train_acc': [], 'test_acc': []}

print("training logistic regression probes...\n")
for layer_idx in tqdm(range(N_LAYERS)):
    lr = LogisticRegression(random_state=42, max_iter=1000)
    lr.fit(X_train_by_layer[layer_idx], y_train)
    
    train_acc = lr.score(X_train_by_layer[layer_idx], y_train)
    test_acc = lr.score(X_test_by_layer[layer_idx], y_test)
    
    lr_results['layers'].append(layer_idx)
    lr_results['train_acc'].append(train_acc)
    lr_results['test_acc'].append(test_acc)

print(f"done | avg test acc: {np.mean(lr_results['test_acc']):.3f}")

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))

ax.plot(dim_results['layers'], dim_results['test_acc'], 'o-', 
        label='difference in means', linewidth=2, markersize=5)
ax.plot(lr_results['layers'], lr_results['test_acc'], 's-', 
        label='logistic regression', linewidth=2, markersize=5)

ax.set_xlabel('layer', fontsize=11)
ax.set_ylabel('test accuracy', fontsize=11)
ax.set_title('gender probe accuracy across layers', fontsize=12)
ax.grid(alpha=0.3)
ax.legend(fontsize=10)
ax.set_ylim([0.4, 1.0])

plt.tight_layout()
plt.savefig('probe_accuracy_by_layer.png', dpi=150, bbox_inches='tight')
plt.show()

print("saved: probe_accuracy_by_layer.png")

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))

x = np.arange(len(dim_results['layers']))
width = 0.2

ax.bar(x - 1.5*width, dim_results['train_acc'], width, label='DIM train', alpha=0.8)
ax.bar(x - 0.5*width, dim_results['test_acc'], width, label='DIM test', alpha=0.8)
ax.bar(x + 0.5*width, lr_results['train_acc'], width, label='LR train', alpha=0.8)
ax.bar(x + 1.5*width, lr_results['test_acc'], width, label='LR test', alpha=0.8)

ax.set_xlabel('layer', fontsize=11)
ax.set_ylabel('accuracy', fontsize=11)
ax.set_title('train vs test accuracy comparison', fontsize=12)
ax.legend(fontsize=9, ncol=2)
ax.set_xticks(x[::5])
ax.set_xticklabels([str(i) for i in dim_results['layers'][::5]])
ax.grid(alpha=0.3, axis='y')
ax.set_ylim([0.4, 1.0])

plt.tight_layout()
plt.savefig('probe_train_vs_test.png', dpi=150, bbox_inches='tight')
plt.show()

print("saved: probe_train_vs_test.png")

In [ ]:
dim_test_acc = np.array(dim_results['test_acc'])
lr_test_acc = np.array(lr_results['test_acc'])

best_dim_layer = np.argmax(dim_test_acc)
best_lr_layer = np.argmax(lr_test_acc)

print("="*50)
print("DIFFERENCE IN MEANS")
print("="*50)
print(f"best layer: {best_dim_layer} (acc={dim_test_acc[best_dim_layer]:.3f})")
print(f"avg: {dim_test_acc.mean():.3f} ± {dim_test_acc.std():.3f}")
print(f"range: {dim_test_acc.min():.3f} - {dim_test_acc.max():.3f}")

print("\n" + "="*50)
print("LOGISTIC REGRESSION")
print("="*50)
print(f"best layer: {best_lr_layer} (acc={lr_test_acc[best_lr_layer]:.3f})")
print(f"avg: {lr_test_acc.mean():.3f} ± {lr_test_acc.std():.3f}")
print(f"range: {lr_test_acc.min():.3f} - {lr_test_acc.max():.3f}")

print("\n" + "="*50)
print("COMPARISON")
print("="*50)
lr_better = np.sum(lr_test_acc > dim_test_acc)
print(f"LR better: {lr_better}/{N_LAYERS} layers")
print(f"avg gap: {(lr_test_acc - dim_test_acc).mean():.4f}")
print(f"max gap: {(lr_test_acc - dim_test_acc).max():.4f}")